# `ab-testing-toolkit` — scratch testing notebook

A running sanity-check sandbox. We add a section here every time a new chunk of the
library is built, so we can eyeball that things work end-to-end (separate from the
automated `pytest` suite and the polished `worked_example.ipynb`).

Run top-to-bottom; every section should execute without error.

## Scaffold smoke test

Confirm the package and its modules import.

In [1]:
import numpy as np

import abtesting
from abtesting import experiment, power, corrections, sequential, visualizations, utils

print("abtesting version:", abtesting.__version__)
print("modules imported:", [m.__name__.split('.')[-1] for m in (experiment, power, corrections, sequential, visualizations, utils)])

abtesting version: 0.1.0
modules imported: ['experiment', 'power', 'corrections', 'sequential', 'visualizations', 'utils']


## `utils`: pre-experiment checks & data cleaning

Sample-ratio-mismatch (SRM) is the *first* thing to check on any experiment: if the
realized split doesn't match the intended one, the randomization/logging is suspect
and the results aren't trustworthy.

In [2]:
from abtesting.utils import check_sample_ratio_mismatch, winsorize, log_transform, cohens_d

healthy = check_sample_ratio_mismatch(10_000, 9_950)
broken = check_sample_ratio_mismatch(10_000, 8_500)
print(f"Healthy split: mismatch={healthy.is_mismatch}  p={healthy.p_value:.3f}")
print(f"Broken split:  mismatch={broken.is_mismatch}  p={broken.p_value:.3e}")

Healthy split: mismatch=False  p=0.723
Broken split:  mismatch=True  p=2.793e-28


In [3]:
# Heavy-tailed metric (e.g. watch time): a few whales blow up the mean.
rng = np.random.default_rng(7)
watch_time = rng.lognormal(mean=2.0, sigma=1.2, size=5_000)

clipped = winsorize(watch_time, lower=0.01, upper=0.99)
logged = log_transform(watch_time)

print(f"raw mean:        {watch_time.mean():8.2f}   max: {watch_time.max():8.2f}")
print(f"winsorized mean: {clipped.mean():8.2f}   max: {clipped.max():8.2f}")
print(f"log1p mean:      {logged.mean():8.2f}   (skew tamed)")

raw mean:           14.77   max:   966.74
winsorized mean:    14.12   max:   118.04
log1p mean:          2.19   (skew tamed)


In [4]:
# Cohen's d: standardized effect size between two arms.
control = rng.normal(0.0, 1.0, 4_000)
treatment = rng.normal(0.3, 1.0, 4_000)
print(f"Cohen's d (true 0.3): {cohens_d(control, treatment):.3f}")

Cohen's d (true 0.3): 0.332


## `Experiment`: hypothesis tests on a single A/B test

The `Experiment` class wraps a control and treatment sample and runs the standard
battery of tests, each returning a uniform `ExperimentResult`.

In [5]:
from abtesting import Experiment

# Continuous metric (e.g. revenue per user) with a true +5 lift.
rng = np.random.default_rng(2024)
control = rng.normal(100, 20, 3_000)
treatment = rng.normal(105, 20, 3_000)

exp = Experiment(control, treatment, metric_type="continuous")
print("summary:", {k: round(v, 3) if isinstance(v, float) else v for k, v in exp.summary().items()})

res = exp.ttest()
print(f"\nWelch t-test: p={res.p_value:.2e}  significant={res.is_significant}")
print(f"  effect size (Cohen's d): {res.effect_size:.3f}")
print(f"  95% CI on mean diff:     ({res.confidence_interval[0]:.2f}, {res.confidence_interval[1]:.2f})")
print(f"  relative lift:           {res.relative_lift:+.1%}")

summary: {'metric_type': 'continuous', 'n_control': 3000, 'n_treatment': 3000, 'control_mean': 99.547, 'treatment_mean': 105.685, 'control_std': 19.776, 'treatment_std': 20.064, 'absolute_effect': 6.137, 'relative_lift': 0.062}

Welch t-test: p=1.87e-32  significant=True
  effect size (Cohen's d): 0.308
  95% CI on mean diff:     (5.13, 7.15)
  relative lift:           +6.2%


In [6]:
# Bootstrap CI cross-checks the parametric t-test interval (no normality assumption).
boot_lo, boot_hi = exp.bootstrap_ci(n_bootstrap=5_000, random_state=1)
print(f"Bootstrap 95% CI on mean diff: ({boot_lo:.2f}, {boot_hi:.2f})")

# Binary / conversion metric: chi-squared proportion test.
c_conv = rng.binomial(1, 0.10, 8_000)
t_conv = rng.binomial(1, 0.12, 8_000)
conv_exp = Experiment(c_conv, t_conv, metric_type="binary")
conv_res = conv_exp.chi_squared()
print(f"\nChi-squared: p={conv_res.p_value:.2e}  significant={conv_res.is_significant}")
print(f"  CI on proportion diff: ({conv_res.confidence_interval[0]:+.3f}, {conv_res.confidence_interval[1]:+.3f})")

# Tidy one-row dataframe for stacking many experiments.
conv_exp.to_dataframe()[["test_name", "p_value", "is_significant", "relative_lift"]]

Bootstrap 95% CI on mean diff: (5.08, 7.12)

Chi-squared: p=2.48e-02  significant=True
  CI on proportion diff: (+0.001, +0.021)


,test_name,p_value,is_significant,relative_lift
0,Chi-squared test,0.02485,True,0.107579


## `power`: pre-experiment planning

Before launching, decide how many users you need and how long that takes. Sample size
grows as the effect you want to detect shrinks, and as you demand more power.

In [7]:
from abtesting import (
    minimum_sample_size,
    minimum_detectable_effect,
    observed_power,
    experiment_runtime_days,
)

baseline = 0.10  # 10% conversion baseline

# Sample size shrinks as we accept a larger detectable effect.
for mde in (0.01, 0.02, 0.05):
    n = minimum_sample_size(baseline, mde, power=0.80)
    days = experiment_runtime_days(20_000, baseline, mde)
    print(f"MDE {mde:+.0%}  ->  {n:>6,} per group  ->  {days} day(s) at 20k/day traffic")

# Inverse: what's the smallest effect we could detect with 50k per arm?
print(f"\nWith 50,000 per arm, MDE = {minimum_detectable_effect(50_000, baseline):.4f} (absolute)")

MDE +1%  ->  14,749 per group  ->  2 day(s) at 20k/day traffic
MDE +2%  ->   3,839 per group  ->  1 day(s) at 20k/day traffic
MDE +5%  ->     683 per group  ->  1 day(s) at 20k/day traffic

With 50,000 per arm, MDE = 0.0053 (absolute)


In [8]:
# Post-hoc power: was our earlier continuous experiment well-powered?
# (effect ~6 absolute on a metric with std ~20, n=3000 per arm)
power_now = observed_power(n=3_000, effect_size=6.0, baseline_std=20.0)
print(f"Observed power of the continuous experiment: {power_now:.3f}")

# A tiny, underpowered version of the same test for contrast.
power_small = observed_power(n=50, effect_size=6.0, baseline_std=20.0)
print(f"Same effect, only n=50 per arm:              {power_small:.3f}  (underpowered)")

Observed power of the continuous experiment: 1.000
Same effect, only n=50 per arm:              0.323  (underpowered)
